# 🚀 SOTA AI: Data Preparation & Advanced Feature Extraction

This self-contained notebook handles the complete data pipeline for the AGI-Frontier Supply Chain & Finance architecture.
It bypasses your local computer entirely—downloading terabytes of raw Kaggle data directly into your Shared Google Drive using Google's datacenter speeds.

It then applies advanced mathematical preprocessing (Topological Homology and Rough Path Theory) to extract precise signal before sharding the data for the 10-node federated training cluster.

In [2]:
from google.colab import drive
import os

# 1. Mount the Shared Google Drive
drive.mount('/content/drive')

# 2. Create the exact folder structure required for the 10x Distributed Strategy
BASE_DIR = '/content/drive/MyDrive/SOTA_Cluster_Shared'
RAW_DIR = os.path.join(BASE_DIR, 'datasets/raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'datasets/processed')
SHARDS_DIR = os.path.join(BASE_DIR, 'datasets/shards')

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SHARDS_DIR, exist_ok=True)

for sub in ['m5', 'paysim', 'dataco', 'nasa_cmapss']:
    os.makedirs(os.path.join(SHARDS_DIR, sub), exist_ok=True)

print(f"\n✅ Drive Mounted. Base directory established at: {BASE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Drive Mounted. Base directory established at: /content/drive/MyDrive/SOTA_Cluster_Shared


## 1. Secure Kaggle Authentication
Run the cell below and paste your `kaggle.json` credentials.
*(Find this by going to Kaggle.com -> Account Settings -> Create New API Token)*

In [3]:
import json
from google.colab import userdata

# Setting up the Kaggle directory
!mkdir -p ~/.kaggle

# Safely input your Kaggle username and key without saving it into the notebook text
import getpass
print("Enter your Kaggle Username:")
username = input()
print("Enter your Kaggle Key:")
key = getpass.getpass()

kaggle_creds = {"username": username, "key": key}

with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(kaggle_creds, file)

!chmod 600 ~/.kaggle/kaggle.json
print("\n✅ Kaggle API Credentials Configured Successfully!")

Enter your Kaggle Username:
nithin1234ag
Enter your Kaggle Key:
··········

✅ Kaggle API Credentials Configured Successfully!


## 2. Install Advanced Mathematical Libraries
Standard `pandas` is not enough. We install `gudhi` for Topological Persistent Homology and `signatory` for Rough Path Theory (Log-Signatures).

In [4]:
# The pre-compiled Signatory wheel is built for Python 3.8 / PyTorch 1.9.0, which crashes Colab's Python 3.10.
# Instead, we leave Colab's default PyTorch 2.x untouched, and compile Signatory from source
# while explicitly disabling the new C++ ABI string format to prevent segfaults during import.

import os
os.environ["_GLIBCXX_USE_CXX11_ABI"] = "0"

!pip install -q gudhi networkx
!pip install -q signatory --no-cache-dir --no-binary signatory

print("\n✅ Advanced Mathematical Libraries Installed (GUDHI, Signatory Source-Compiled, NetworkX)!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done

✅ Advanced Mathematical Libraries Installed (GUDHI, Signatory, NetworkX)!


## 3. Download the 4 Core Datasets (Direct to Google Drive)
This pulls the gigabytes of data directly into the specified Drive folders using API datacenter speeds.

In [5]:
import os
BASE_DIR = '/content/drive/MyDrive/SOTA_Cluster_Shared'
RAW_DIR = os.path.join(BASE_DIR, 'datasets/raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'datasets/processed')
SHARDS_DIR = os.path.join(BASE_DIR, 'datasets/shards')

import os
os.chdir(RAW_DIR)

# 1. DataCo SMART Supply Chain Dataset (Topology)
!kaggle datasets download -d shashwatwork/dataco-smart-supply-chain-for-big-data-analysis --unzip

# 2. PaySim Synthetic Financial/Fraud Dataset (Topology/Finance)
!kaggle datasets download -d ealaxi/paysim1 --unzip

# 3. M5 Forecasting Walmart Dataset (Demand Forecaster)
!kaggle competitions download -c m5-forecasting-accuracy
!unzip -q m5-forecasting-accuracy.zip -d m5_data
!rm m5-forecasting-accuracy.zip

# 4. NASA Turbofan IoT Telemetry (Mamba sequences)
!kaggle datasets download -d behrad3d/nasa-cmaps --unzip

print("\n✅ All 4 Datasets Downloaded and Unzipped to Shared Google Drive!")

Dataset URL: https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis
License(s): CC0-1.0
100% 25.7M/25.7M [00:00<00:00, 267MB/s]
100% 25.7M/25.7M [00:00<00:00, 256MB/s]
Dataset URL: https://www.kaggle.com/datasets/ealaxi/paysim1
License(s): CC-BY-SA-4.0
 97% 173M/178M [00:01<00:00, 64.5MB/s]
100% 178M/178M [00:01<00:00, 106MB/s] 
403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/competitions/data/download-all/m5-forecasting-accuracy
unzip:  cannot find or open m5-forecasting-accuracy.zip, m5-forecasting-accuracy.zip.zip or m5-forecasting-accuracy.zip.ZIP.
rm: cannot remove 'm5-forecasting-accuracy.zip': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/behrad3d/nasa-cmaps
License(s): CC0-1.0
  0% 0.00/12.3M [00:00<?, ?B/s]
100% 12.3M/12.3M [00:00<00:00, 186MB/s]

✅ All 4 Datasets Downloaded and Unzipped to Shared Google Drive!


## 4. Advanced Preprocessing: Continuous-Time Log-Signatures
Instead of moving averages (which destroy high-frequency IoT vibration anomalies), we compress the NASA streaming sequences using **Rough Path Theory**.

In [ ]:
import os
BASE_DIR = '/content/drive/MyDrive/SOTA_Cluster_Shared'
RAW_DIR = os.path.join(BASE_DIR, 'datasets/raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'datasets/processed')
SHARDS_DIR = os.path.join(BASE_DIR, 'datasets/shards')

import torch
import signatory
import pandas as pd
import numpy as np
import glob
import os
import gc

def extract_log_signatures(sensor_tensor, depth=3):
    """Extracts the Log-Signature of a continuous path."""
    with torch.no_grad():
        return signatory.logsignature(sensor_tensor, depth)

print("Searching for NASA CMAPSS Datasets...")
nasa_files = glob.glob(os.path.join(RAW_DIR, '**', 'train_FD*.txt'), recursive=True)
if not nasa_files:
    nasa_files = glob.glob(os.path.join(RAW_DIR, 'train_FD*.txt'))

if nasa_files:
    print(f"Found {len(nasa_files)} NASA files. Processing real data with immediate saving...")
    for file_path in nasa_files:
        filename = os.path.basename(file_path)
        print(f"\n-> Loading {filename} into memory...")
        try:
            df = pd.read_csv(file_path, sep='\\s+', header=None)
            df = df.dropna(axis=1, how='all')
            engine_ids = df.iloc[:, 0].unique()
            
            out_name = f"features_logsig_{filename.replace('.txt', '.csv')}"
            out_path = os.path.join(PROCESSED_DIR, out_name)
            
            # Overwrite any existing file from a previous crashed run
            if os.path.exists(out_path):
                os.remove(out_path)
            
            print(f"-> Streaming Log-Signatures to {out_path} engine-by-engine...")
            
            for eng_id in engine_ids:
                engine_data = df[df.iloc[:, 0] == eng_id].iloc[:, 2:].values 
                tensor_data = torch.tensor(engine_data, dtype=torch.float32).unsqueeze(0) 
                
                if tensor_data.shape[1] > 1:
                    sig = extract_log_signatures(tensor_data, depth=2)
                    sig_array = sig.squeeze(0).numpy()
                    
                    # 1. Format this exact engine into a 1-row DataFrame
                    row_dict = {f"sig_{i}": val for i, val in enumerate(sig_array)}
                    row_dict['engine_id'] = eng_id
                    row_df = pd.DataFrame([row_dict])
                    
                    # 2. Append directly to Google Drive CSV immediately
                    # Write header only if the file does not exist yet
                    row_df.to_csv(out_path, mode='a', header=not os.path.exists(out_path), index=False)
                
                del tensor_data, engine_data
            
            del df
            gc.collect()
            print(f"   ✅ Successfully saved all {len(engine_ids)} engines for {filename}")
            
        except Exception as e:
            print(f"   ❌ Error processing {filename}: {e}")
else:
    print("⚠️ Could not locate NASA CMAPSS files to process.")
    
print("\n✅ Log-Signature Feature Streaming Complete!")

## 5. Advanced Preprocessing: Topological Deep Learning
We don't want a flat ledger. We extract the Betti numbers (physical holes/choke-points in the supply chain flow) using Gudhi.

In [ ]:
import os
BASE_DIR = '/content/drive/MyDrive/SOTA_Cluster_Shared'
RAW_DIR = os.path.join(BASE_DIR, 'datasets/raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'datasets/processed')
SHARDS_DIR = os.path.join(BASE_DIR, 'datasets/shards')

import gudhi
import networkx as nx
import numpy as np
import pandas as pd
import glob
import os

def extract_betti_numbers(distance_matrix):
    """Extracts Betti numbers (Shape features) using Persistent Homology."""
    rips_complex = gudhi.RipsComplex(distance_matrix=distance_matrix, max_edge_length=2.0)
    simplex_tree = rips_complex.create_simplex_tree(max_dimension=3)
    simplex_tree.persistence()
    return simplex_tree.betti_numbers()

print("Searching for PaySim / DataCo Fraud Datasets for Graph Extraction...")
paysim_files = glob.glob(os.path.join(RAW_DIR, '*paysim*.csv')) + glob.glob(os.path.join(RAW_DIR, '*PS*.csv')) + glob.glob(os.path.join(RAW_DIR, '**', 'PS*.csv'), recursive=True)

if paysim_files:
    file_path = paysim_files[0]
    print(f"\n-> Processing Topological Features for: {os.path.basename(file_path)}")
    
    out_path = os.path.join(PROCESSED_DIR, "topological_fraud_features.csv")
    if os.path.exists(out_path):
        os.remove(out_path)
    
    print(f"-> Streaming 50,000-row chunks directly to {out_path}...")
    
    try:
        chunk_index = 0
        # The full PaySim dataset is 6.3 Million rows. We process sequentially to prevent RAM crashes.
        for chunk_df in pd.read_csv(file_path, chunksize=50000):
            chunk_index += 1
            print(f"   Processing Chunk {chunk_index} (Rows {(chunk_index-1)*50000} to {chunk_index*50000})...")
            
            G = nx.from_pandas_edgelist(chunk_df, 'nameOrig', 'nameDest', ['amount'], create_using=nx.Graph())
            components = sorted(nx.connected_components(G), key=len, reverse=True)
            
            if len(components) > 0:
                sub_G = G.subgraph(components[0])
                nodes = list(sub_G.nodes())[:150] # Hard bound to prevent RipsComplex OOM
                sub_G = sub_G.subgraph(nodes)
                
                dist_matrix = np.zeros((len(nodes), len(nodes)))
                for i, n1 in enumerate(nodes):
                    for j, n2 in enumerate(nodes):
                        if i != j:
                            if sub_G.has_edge(n1, n2):
                                dist_matrix[i, j] = 1.0 / (sub_G[n1][n2]['amount'] + 1e-5)
                            else:
                                dist_matrix[i, j] = 10.0
                
                dist_matrix = (dist_matrix + dist_matrix.T) / 2
                np.fill_diagonal(dist_matrix, 0)
                
                betti = extract_betti_numbers(dist_matrix)
                
                # Append immediately to Google Drive
                feature_df = pd.DataFrame([{
                    "chunk_id": chunk_index,
                    "betti_0_connected_comps": betti[0] if len(betti) > 0 else 0, 
                    "betti_1_rings": betti[1] if len(betti) > 1 else 0,
                    "betti_2_voids": betti[2] if len(betti) > 2 else 0
                }])
                
                feature_df.to_csv(out_path, mode='a', header=not os.path.exists(out_path), index=False)
                
    except Exception as e:
        print(f"   ❌ Error processing PaySim topology chunk: {e}")
else:
    print("⚠️ Could not locate PaySim/Fraud file to process.")

print("\n✅ Topological Streaming Complete!")

## 6. Sharding for the 10-Node Federated Cluster
To prevent Colab's 12GB RAM limit from crashing during training, we actively split the heavy datasets (like PaySim's 6 Million rows) into 10 smaller chunks.

In [ ]:
import pandas as pd
import glob

def shard_dataset(csv_path, output_folder, num_shards=10):
    print(f"Loading massive dataset: {csv_path}")
    # For extremely large files, it is safer to read entirely and split,
    # or use chunksize in pandas for low-RAM streaming. We use simple split here.
    df = pd.read_csv(csv_path)

    chunk_size = len(df) // num_shards
    print(f"Total rows: {len(df)}. Sharding into {num_shards} files of ~{chunk_size} rows each...")

    for i in range(num_shards):
        start_idx = i * chunk_size
        # Last shard gets the remaining rows
        end_idx = (i + 1) * chunk_size if i < num_shards - 1 else len(df)

        shard_df = df.iloc[start_idx:end_idx]

        shard_filename = os.path.join(output_folder, f"shard_{i+1}.csv")
        shard_df.to_csv(shard_filename, index=False)
        print(f"Saved {shard_filename}")

# Example Execution on PaySim (This will execute against the downloaded CSV)
# Find the correct filename
paysim_csvs = glob.glob(os.path.join(RAW_DIR, '*.csv'))
paysim_target = [f for f in paysim_csvs if 'simulated' in f.lower() or 'paysim' in f.lower()]

if paysim_target:
    shard_dataset(paysim_target[0], os.path.join(SHARDS_DIR, 'paysim'))
else:
    print("\n⚠️ Could not locate PaySim dataset block. (Will proceed dynamically). ")

print("\n✅ Datasets sharded and distributed for the 10x Colab Training Loop!")